# Fine-tune BERT for Amazon Review Sentiment

Notebook này dùng Google Colab GPU để fine-tune `bert-base-uncased` cho bài toán sentiment 3 lớp: `Negative`, `Neutral`, `Positive`.

Trước khi chạy: vào `Runtime` -> `Change runtime type` -> chọn `T4 GPU` hoặc GPU tương đương.

In [ ]:
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Install Dependencies

Colab thường đã có PyTorch. Cell này cài thêm thư viện cần thiết cho Hugging Face và evaluation.

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib tqdm safetensors

## 2. Upload Dataset

Upload file `data/processed/labeled_reviews.csv` từ máy của bạn. Sau khi upload, notebook sẽ lưu đúng cấu trúc `data/processed/labeled_reviews.csv`.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

Path('data/processed').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    target = Path('data/processed/labeled_reviews.csv')
    shutil.move(filename, target)
    print('Saved dataset to:', target)
    break

## 3. Imports And Config

Các hyperparameters được để ở một chỗ để dễ chỉnh. Nếu Colab bị out-of-memory, giảm `BATCH_SIZE` xuống `8`.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_scheduler

INPUT_PATH = 'data/processed/labeled_reviews.csv'
MODEL_NAME = 'bert-base-uncased'
MODEL_DIR = Path('models/bert')
OUTPUT_DIR = Path('outputs/bert')

MAX_LEN = 128
EPOCHS = 2
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42

LABEL_ORDER = ['Negative', 'Neutral', 'Positive']
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_ORDER)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_random_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 4. Load And Split Data

Map label text sang số để BERT train classification 3 lớp.

In [ ]:
df = pd.read_csv(INPUT_PATH)
df = df[['reviewText', 'sentiment_label']].dropna().copy()
df['reviewText'] = df['reviewText'].astype(str).str.strip()
df['sentiment_label'] = df['sentiment_label'].astype(str).str.strip()
df = df[df['reviewText'] != '']
df = df[df['sentiment_label'].isin(LABEL_ORDER)]
df['label_id'] = df['sentiment_label'].map(LABEL2ID)

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['reviewText'],
    df['label_id'],
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df['label_id'],
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_labels,
)

print('Train:', len(train_texts))
print('Val:', len(val_texts))
print('Test:', len(test_texts))
print(df['sentiment_label'].value_counts())

## 5. Tokenizer And Dataset

`max_length=128` giúp giảm RAM/GPU memory nhưng vẫn đủ hợp lý cho review ngắn-vừa.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ReviewDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = ReviewDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 6. Build Model

BERT classification head sẽ được train lại cho 3 lớp sentiment.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_ORDER),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_labels.to_numpy())
class_weights = torch.tensor(weights, dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
print('Class weights:', weights)

## 7. Train And Validate

Model tốt nhất được chọn theo validation macro F1-score và lưu vào `models/bert/`.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

def train_one_epoch():
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc='Training'):
        optimizer.zero_grad(set_to_none=True)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def predict(loader):
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            preds = torch.argmax(outputs.logits, dim=1)
            total_loss += outputs.loss.item()
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    return np.array(y_true), np.array(y_pred), total_loss / len(loader)

history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_macro_f1': []}
best_macro_f1 = -1

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    train_loss = train_one_epoch()
    y_val_true, y_val_pred, val_loss = predict(val_loader)
    val_acc = accuracy_score(y_val_true, y_val_pred)
    val_f1 = f1_score(y_val_true, y_val_pred, average='macro', zero_division=0)

    history['train_loss'].append(float(train_loss))
    history['val_loss'].append(float(val_loss))
    history['val_accuracy'].append(float(val_acc))
    history['val_macro_f1'].append(float(val_f1))

    print(f'Train loss: {train_loss:.4f}')
    print(f'Val loss: {val_loss:.4f}')
    print(f'Val accuracy: {val_acc:.4f}')
    print(f'Val macro F1: {val_f1:.4f}')

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print('Saved best model to', MODEL_DIR)

with open(MODEL_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

## 8. Evaluate Test Set

In accuracy, macro F1-score, confusion matrix và classification report.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)

y_true, y_pred, test_loss = predict(test_loader)
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
report = classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=LABEL_ORDER, zero_division=0)

print('Test loss:', round(test_loss, 4))
print('Accuracy:', round(accuracy, 4))
print('Macro F1:', round(macro_f1, 4))
print('\nConfusion matrix:')
print(pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER))
print('\nClassification report:')
print(report)

metrics = {'test_loss': float(test_loss), 'accuracy': float(accuracy), 'macro_f1': float(macro_f1)}
with open(OUTPUT_DIR / 'bert_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
with open(OUTPUT_DIR / 'bert_classification_report.txt', 'w') as f:
    f.write(report)

## 9. Save Confusion Matrix Chart

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('BERT Confusion Matrix')
plt.colorbar()
plt.xticks(range(len(LABEL_ORDER)), LABEL_ORDER, rotation=45, ha='right')
plt.yticks(range(len(LABEL_ORDER)), LABEL_ORDER)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

threshold = cm.max() / 2 if cm.max() else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > threshold else 'black'
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bert_confusion_matrix.png', dpi=150)
plt.show()

## 10. Download Outputs

Nén model và outputs để tải về máy.

In [ ]:
!zip -r bert_results.zip models/bert outputs/bert
files.download('bert_results.zip')